# Site build: data for the website (Phase 3)

This notebook only **reads** the existing files and writes **new** files into `site/data/`.
It never changes anything in `data/`, `img/` or on Drive.

Inputs:
- from the repo: `data/matches_2026.csv`, `data/rain_affected_matches.csv`, `data/replay/*.json`, `data/worst_balls_death_2026.json`
- from Drive (read-only): `bundle_2026.parquet` (the frozen 2026 predictions), `site_extract_2026.parquet` (raw scorecard columns for 2026, made by the one extract cell) and `balls_2026_explained.parquet` (venues)

Outputs (new): `site/data/matches_2026.json` (match number, teams, venue for every 2026 match) and `site/data/scorecards_2026.json` (the scorecard before each replay ball and each worst ball).

Run it in Colab (it mounts Drive) or anywhere with the Drive files in a folder (set `SITE_DRIVE`).

In [1]:
# ===== S0. Setup (COPY-PASTE) =====
import json, os, glob
import numpy as np
import pandas as pd

if os.path.exists('/content'):                       # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
    REPO = '/content/ml_death_overs_ipl/'            # clone the repo here first
else:
    DRIVE = os.environ.get('SITE_DRIVE', 'drive/')
    REPO = os.environ.get('SITE_REPO', '../')
OUT = REPO + 'site/data/'
os.makedirs(OUT, exist_ok=True)

EXTRACT = DRIVE + 'site_extract_2026.parquet'
HAVE_EXTRACT = os.path.exists(EXTRACT)
print('extract found:', HAVE_EXTRACT)

extract found: True


## S1. Match numbers

ESPNcricinfo's 2026 match ids are consecutive in three blocks: Matches 1-20, Matches 21-70, then the four playoffs (Qualifier 1, Eliminator, Qualifier 2, Final). The ids missing from `matches_2026.csv` are exactly the three rain-affected 2026 matches, so every match keeps its official number. Check: KKR vs LSG (the tie decided by a super over) is Match 38.

In [2]:
# ===== S1. Match numbers, teams and venues (COPY-PASTE) =====
scored = pd.read_csv(REPO + 'data/matches_2026.csv')
rain = pd.read_csv(REPO + 'data/rain_affected_matches.csv')['match_id']
BLOCKS = [(1527674, 1, 20), (1529264, 21, 50)]       # (first id, first match number, matches in block)
PLAYOFFS = {1535462: 'Qualifier 1', 1535463: 'Eliminator', 1535464: 'Qualifier 2', 1535465: 'Final'}

def number(mid):
    for first, no, n in BLOCKS:
        if first <= mid < first + n:
            return mid - first + no
    return None

ids = sorted(set(scored['match_id']) | {m for m in rain if number(m) or m in PLAYOFFS})
M = {}
for mid in ids:
    no = number(mid)
    M[mid] = {'no': no if no else 70 + 1 + list(PLAYOFFS).index(mid),
              'label': f'Match {no}' if no else PLAYOFFS[mid],
              'scored': bool(mid in set(scored['match_id'])), 'rain': bool(mid in set(rain))}

print('1. matches:', len(M), '| scored:', sum(m['scored'] for m in M.values()),
      '| rain:', sum(m['rain'] for m in M.values()))                                   # expect 74 | 71 | 3
print('2. numbers 1-74, each once:', sorted(m['no'] for m in M.values()) == list(range(1, 75)))  # expect True
print('3. KKR vs LSG tie:', M[1529281]['label'])                                         # expect Match 38

# teams and venue
if HAVE_EXTRACT:
    X = pd.read_parquet(EXTRACT)
    for mid, g in X.groupby('match_id'):
        if mid not in M: continue
        bat = g.sort_values(['innings']).groupby('innings')['batting_team'].first()
        M[mid]['teams'] = [str(t) for t in bat.values[:2]]
        if len(bat) == 1:                                                  # one innings only (rain)
            M[mid]['teams'].append(str(g['bowling_team'].iloc[0]))
        M[mid]['venue'] = str(g['venue'].iloc[0])
    seq = X.groupby('match_id')['match_seq'].min().rank().astype(int)
    print('4. match_seq order agrees with the numbers:',
          all(seq[m] == M[m]['no'] for m in seq.index if m in M))                       # expect True
else:
    E = pd.read_parquet(DRIVE + 'balls_2026_explained.parquet')
    for mid, v in E.groupby('match_id')['venue'].first().items():
        if mid in M: M[mid]['venue'] = str(v)
    for p in glob.glob(REPO + 'data/replay/1*.json'):                       # teams we already know
        R = json.load(open(p))
        M[R['match_id']]['teams'] = [i['batting'] for i in R['innings']]
    print('4. no extract yet: teams only for the replay matches')

# per-match log-loss for the Extras page, all balls and death balls, from the frozen predictions
Bn = pd.read_parquet(DRIVE + 'bundle_2026.parquet')
yb = Bn['y'].to_numpy().astype(int); ib = np.arange(len(yb))
lx = -np.log(np.clip(Bn[[f'px_{k}' for k in range(7)]].to_numpy()[ib, yb], 1e-15, 1))
lb = -np.log(np.clip(Bn[[f'pb1_{k}' for k in range(7)]].to_numpy()[ib, yb], 1e-15, 1))
G = pd.DataFrame({'match_id': Bn['match_id'], 'death': Bn['death'].astype(bool), 'lx': lx, 'd': lx - lb})
for sl, g in [('all', G), ('death', G[G['death']])]:
    a = g.groupby('match_id').agg(n=('lx', 'size'), ll=('lx', 'mean'), gain=('d', 'mean'))
    for mid, r in a.iterrows():
        M[int(mid)].update({f'balls_{sl}': int(r['n']), f'll_xgb_{sl}': round(float(r['ll']), 4), f'gain_vs_b1_{sl}': round(float(r['gain']), 4)})
chk = scored.set_index('match_id')
print('5. all-ball numbers match matches_2026.csv:',
      all(abs(M[m]['ll_xgb_all'] - chk.at[m, 'LX']) < 1e-4 and M[m]['balls_all'] == chk.at[m, 'balls'] for m in chk.index))  # expect True
print('6. death balls in all:', sum(M[m].get('balls_death', 0) for m in M), '| death log-loss:',
      round(G.loc[G['death'], 'lx'].mean(), 4))                                          # expect 3583 | 1.568
with open(OUT + 'matches_2026.json', 'w') as f:
    json.dump({str(k): v for k, v in M.items()}, f, indent=1)
print('saved', OUT + 'matches_2026.json')

1. matches: 74 | scored: 71 | rain: 3
2. numbers 1-74, each once: True
3. KKR vs LSG tie: Match 38
4. match_seq order agrees with the numbers: True


5. all-ball numbers match matches_2026.csv: True
6. death balls in all: 3583 | death log-loss: 1.568
saved ../site/data/matches_2026.json


## S2. Scorecards

For every replay ball and every worst ball: the batter's runs and balls, the non-striker's, and the bowler's figures, all **before** the ball (the situation the models saw). The over label is built the same way as in `site_export.ipynb` (legal balls bowled before the ball), and wides and no-balls share the label of the next legal ball, so the join uses (match, innings, over label, order within the label).

In [3]:
# ===== S2. Scorecards before each ball (COPY-PASTE) =====
def overs(balls):
    return f'{int(balls) // 6}.{int(balls) % 6}'

if HAVE_EXTRACT:
    X = X.sort_values('row').reset_index(drop=True) if 'row' in X else X
    X = X.sort_values(['match_id', 'innings', 'over', 'ball'], kind='stable').reset_index(drop=True)
    b = X['c_team_balls'].astype(int)
    X['label'] = (b // 6).astype(str) + '.' + (b % 6 + 1).astype(str)
    X['k'] = X.groupby(['match_id', 'innings', 'label']).cumcount()
    X['key'] = X['match_id'].astype(str) + '|' + X['innings'].astype(int).astype(str) + '|' + X['label'] + '|' + X['k'].astype(str)
    X = X.set_index('key')

    want = {}                                               # key -> (batter, bowler) as the site files have them
    for p in glob.glob(REPO + 'data/replay/1*.json'):
        R = json.load(open(p))
        for inn in R['innings']:
            seen = {}
            for ball in inn['balls']:
                k = seen.get(ball['over'], 0); seen[ball['over']] = k + 1
                want[f"{R['match_id']}|{inn['innings']}|{ball['over']}|{k}"] = (ball['batter'], ball['bowler'])
    W = json.load(open(REPO + 'data/worst_balls_death_2026.json'))
    for ball in W['balls']:                                 # worst balls are legal: the last row with that label
        g = X.loc[X.index.str.startswith(f"{ball['match_id']}|{ball['innings']}|{ball['over']}|")]
        g = g[g['valid_ball'] == 1] if 'valid_ball' in g else g
        want[g.index[-1]] = (ball['batter'], ball['bowler'])

    C, bad = {}, []
    for key, (bat, bowl) in want.items():
        r = X.loc[key]
        if r['batter'] != bat or r['bowler'] != bowl:
            bad.append(key)
        C[key] = {'batting': str(r['batting_team']), 'bowling': str(r['bowling_team']),
                  'bat': [str(r['batter']), int(r['c_batter_runs']), int(r['c_batter_balls'])],
                  'ns': [str(r['non_striker']), int(r['non_striker_runs']), int(r['non_striker_balls'])],
                  'bowl': [str(r['bowler']), overs(r['c_bowler_balls']), int(r['c_bowler_runs']), int(r['c_bowler_wickets'])]}
    print('1. balls with a scorecard:', len(C), '| batter or bowler mismatches:', len(bad))     # expect ~245 | 0
    k = '1529281|2|19.6|0'
    print('2. last ball of the tie:', C.get(k))         # Shami batting, Kartik Tyagi bowling
    with open(OUT + 'scorecards_2026.json', 'w') as f:
        json.dump(C, f, separators=(',', ':'))
    print('saved', OUT + 'scorecards_2026.json')
else:
    print('skipped: run the extract cell first (it makes site_extract_2026.parquet)')

1. balls with a scorecard: 258 | batter or bowler mismatches: 0
2. last ball of the tie: {'batting': 'Lucknow Super Giants', 'bowling': 'Kolkata Knight Riders', 'bat': ['Mohammed Shami', 5, 5], 'ns': ['Prince Yadav', 0, 1], 'bowl': ['Kartik Tyagi', '3.5', 35, 1]}
saved ../site/data/scorecards_2026.json
